In [38]:
# ------------------------------------------------------------
# STEP 1: LOAD DATA
# ------------------------------------------------------------

import pandas as pd
import numpy as np

url = "gs://agntworks-data-dev/sandbox/experiments/clean_flight_data_with_regime_v3.csv"

df = pd.read_csv(url, parse_dates=["dep_datetime"])

print("Shape:", df.shape)
df.head()

Shape: (10794, 12)


,Trip_Number,Trip_Legs_ID,dep_date,dep_datetime,Dep_Time_Actual_GMT,Dep_cluster,Arr_cluster,Quote_Total_Cost,Revenue_allocated,Block_Hours,RevHr_clean,regime
0,157565,42366991KODR3217,2016-01-01,2016-01-01 23:37:00,23:37:00,WASHINGTON_DC_CLUSTER,OTHER_CLUSTER,44251.20,11954.115640,2.850000,4194.426540,NORMAL
1,157565,420331372FOSJ3517,2016-01-02,2016-01-02 18:00:00,18:00:00,OTHER_CLUSTER,MIAMI_CLUSTER,44251.20,6291.639810,1.500000,4194.426540,PEAK_MIXED
2,157565,420331372FOSJ4217,2016-01-02,2016-01-02 20:42:00,20:42:00,MIAMI_CLUSTER,SAN_FRANCISCO_CLUSTER,44251.20,26005.444550,6.200000,4194.426540,PEAK_MIXED
3,162215,42125996CASG317,2016-01-10,2016-01-10 14:56:00,14:56:00,BOSTON_CLUSTER,OTHER_CLUSTER,76864.32,22011.546392,4.166667,5282.771134,HIGH_YIELD
4,162215,423761209HARN017,2016-01-10,2016-01-10 21:10:00,21:10:00,OTHER_CLUSTER,SAN_JUAN_CLUSTER,76864.32,6251.279175,1.183333,5282.771134,HIGH_YIELD


In [39]:
# ------------------------------------------------------------
# STEP 2: VERIFY TIME FIELDS EXIST AND ARE VALID
# ------------------------------------------------------------

required_cols = ["dep_datetime"]

missing = [c for c in required_cols if c not in df.columns]
print("Missing time columns:", missing)

# Null check
print("Null dep_datetime:", df["dep_datetime"].isna().sum())

# Hour distribution
df["hour"] = df["dep_datetime"].dt.hour

print("\nHour summary:")
print(df["hour"].describe())

print("\nHour distribution:")
display(df["hour"].value_counts().sort_index())

Missing time columns: []
Null dep_datetime: 0

Hour summary:
count    10794.000000
mean        15.970817
std          5.595873
min          0.000000
25%         14.000000
50%         17.000000
75%         20.000000
max         23.000000
Name: hour, dtype: float64

Hour distribution:


hour
0      352
1      251
2      150
3      119
4       66
5       53
6       34
7        5
8       17
9       11
10      66
11     187
12     388
13     548
14     730
15     881
16     927
17    1006
18    1035
19    1035
20     917
21     853
22     685
23     478
Name: count, dtype: int64

In [40]:
# ------------------------------------------------------------
# STEP 2.5: REMOVE INTRA-CLUSTER (REPO NOISE)
# ------------------------------------------------------------

initial_rows = len(df)

# 1) Remove same-cluster legs (e.g., NY→NY, SF→SF)
df = df[df["Dep_cluster"] != df["Arr_cluster"]].copy()

print("Removed intra-cluster rows:", initial_rows - len(df))
print("Remaining rows:", len(df))

# ------------------------------------------------------------
# OPTIONAL: REMOVE GENERIC / LOW-SIGNAL CLUSTERS
# (uncomment if needed)
# ------------------------------------------------------------

# bad_clusters = ["OTHER_CLUSTER"]
# df = df[
#     (~df["Dep_cluster"].isin(bad_clusters)) &
#     (~df["Arr_cluster"].isin(bad_clusters))
# ].copy()

# print("After removing OTHER clusters:", len(df))

Removed intra-cluster rows: 1262
Remaining rows: 9532


In [44]:
# ------------------------------------------------------------
# STEP 2.6: VERIFY NO INTRA-CLUSTER LEFT
# ------------------------------------------------------------

same_cluster = df[df["Dep_cluster"] == df["Arr_cluster"]]

print("Remaining same-cluster rows:", len(same_cluster))

# Should be 0
same_cluster.head()

Remaining same-cluster rows: 0


,Trip_Number,Trip_Legs_ID,dep_date,dep_datetime,Dep_Time_Actual_GMT,Dep_cluster,Arr_cluster,Quote_Total_Cost,Revenue_allocated,Block_Hours,RevHr_clean,regime,hour


In [48]:
# ------------------------------------------------------------
# STEP 3: REBUILD TIME FEATURES (dow, hour, tod)
# ------------------------------------------------------------

# Hour
df["hour"] = df["dep_datetime"].dt.hour

# Day of week
df["dow"] = df["dep_datetime"].dt.day_name()

# Time-of-day buckets
def get_tod(hour):
    if 5 <= hour < 10:
        return "MORNING"
    elif 10 <= hour < 15:
        return "MIDDAY"
    elif 15 <= hour < 20:
        return "EVENING"
    else:
        return "NIGHT"

df["tod"] = df["hour"].apply(get_tod)

# Sanity check
print("Null dow:", df["dow"].isna().sum())
print("Null tod:", df["tod"].isna().sum())

df[["dep_datetime", "dow", "hour", "tod"]].head()

Null dow: 0
Null tod: 0


,dep_datetime,dow,hour,tod
0,2016-01-01 23:37:00,Friday,23,NIGHT
1,2016-01-02 18:00:00,Saturday,18,EVENING
2,2016-01-02 20:42:00,Saturday,20,NIGHT
3,2016-01-10 14:56:00,Sunday,14,MIDDAY
4,2016-01-10 21:10:00,Sunday,21,NIGHT


In [41]:
# ---------------------------------------------
# REGIME DISTRIBUTION WITH % OF TOTAL
# ---------------------------------------------

# Count per regime
regime_counts = df["regime"].value_counts().reset_index()
regime_counts.columns = ["regime", "count"]

# Calculate percentage
total_rows = len(df)
regime_counts["percent_of_total"] = (regime_counts["count"] / total_rows) * 100

# Format percentage for readability
regime_counts["percent_of_total"] = regime_counts["percent_of_total"].round(2)

# Sort for clean view (optional)
regime_counts = regime_counts.sort_values(by="count", ascending=False)

# Display
regime_counts

,regime,count,percent_of_total
0,NORMAL,5560,58.33
1,HIGH_YIELD,1604,16.83
2,PEAK_MIXED,1306,13.70
3,HIGH_VOLUME,1062,11.14


In [46]:
# ---------------------------------------------
# REGIME DISTRIBUTION + REVENUE CONTRIBUTION
# ---------------------------------------------

# Total rows + total revenue
total_rows = len(df)
total_revenue = df["Quote_Total_Cost"].sum()

# Aggregate by regime
regime_summary = (
    df.groupby("regime")
      .agg(
          count=("regime", "size"),
          total_revenue=("Quote_Total_Cost", "sum")
      )
      .reset_index()
)

# % of total rows
regime_summary["percent_of_rows"] = (
    regime_summary["count"] / total_rows * 100
)

# % of total revenue
regime_summary["percent_of_revenue"] = (
    regime_summary["total_revenue"] / total_revenue * 100
)

# Formatting
regime_summary["percent_of_rows"] = regime_summary["percent_of_rows"].round(2)
regime_summary["percent_of_revenue"] = regime_summary["percent_of_revenue"].round(2)

# Sort by revenue contribution
regime_summary = regime_summary.sort_values(
    by="total_revenue", ascending=False
)

regime_summary

,regime,count,total_revenue,percent_of_rows,percent_of_revenue
2,NORMAL,5560,2.022296e+08,58.33,53.40
1,HIGH_YIELD,1604,7.441071e+07,16.83,19.65
3,PEAK_MIXED,1306,6.140289e+07,13.70,16.21
0,HIGH_VOLUME,1062,4.068943e+07,11.14,10.74


In [49]:
# ------------------------------------------------------------
# STEP 3: CREATE DOW + TOD
# ------------------------------------------------------------

# Day of week
df["dow"] = df["dep_datetime"].dt.day_name()

# Time-of-day buckets
def get_tod(hour):
    if 5 <= hour < 10:
        return "MORNING"
    elif 10 <= hour < 15:
        return "MIDDAY"
    elif 15 <= hour < 20:
        return "EVENING"
    else:
        return "NIGHT"

df["tod"] = df["hour"].apply(get_tod)

# Quick check
df[["dep_datetime", "dow", "hour", "tod"]].head()

,dep_datetime,dow,hour,tod
0,2016-01-01 23:37:00,Friday,23,NIGHT
1,2016-01-02 18:00:00,Saturday,18,EVENING
2,2016-01-02 20:42:00,Saturday,20,NIGHT
3,2016-01-10 14:56:00,Sunday,14,MIDDAY
4,2016-01-10 21:10:00,Sunday,21,NIGHT


In [50]:
# ------------------------------------------------------------
# STEP 4: DEFINE SEGMENT
# ------------------------------------------------------------

df["segment"] = (
    df["Dep_cluster"] + "→" +
    df["Arr_cluster"] + "|" +
    df["dow"] + "|" +
    df["tod"]
)

print("Unique segments:", df["segment"].nunique())
df["segment"].head()

Unique segments: 4375


0     WASHINGTON_DC_CLUSTER→OTHER_CLUSTER|Friday|NIGHT
1         OTHER_CLUSTER→MIAMI_CLUSTER|Saturday|EVENING
2    MIAMI_CLUSTER→SAN_FRANCISCO_CLUSTER|Saturday|N...
3           BOSTON_CLUSTER→OTHER_CLUSTER|Sunday|MIDDAY
4          OTHER_CLUSTER→SAN_JUAN_CLUSTER|Sunday|NIGHT
Name: segment, dtype: object

In [56]:
# ------------------------------------------------------------
# STEP: CHECK BASE DENSITY (NO DOW, NO TOD)
# ------------------------------------------------------------

df["corridor"] = df["Dep_cluster"] + "→" + df["Arr_cluster"]

corridor_stats = (
    df.groupby("corridor")
      .agg(
          flights=("Trip_Legs_ID", "count"),
          total_revenue=("Revenue_allocated", "sum")
      )
      .reset_index()
      .sort_values("flights", ascending=False)
)

corridor_stats.head(20)

,corridor,flights,total_revenue
292,MIAMI_CLUSTER→NEW_YORK_CLUSTER,293,4.539167e+06
341,NEW_YORK_CLUSTER→MIAMI_CLUSTER,241,3.732947e+06
60,BOSTON_CLUSTER→NEW_YORK_CLUSTER,178,1.130057e+06
270,LOS_ANGELES_CLUSTER→SAN_FRANCISCO_CLUSTER,170,1.364936e+06
329,NEW_YORK_CLUSTER→BOSTON_CLUSTER,169,1.037140e+06
489,SAN_FRANCISCO_CLUSTER→LOS_ANGELES_CLUSTER,165,1.230083e+06
493,SAN_FRANCISCO_CLUSTER→NEW_YORK_CLUSTER,128,3.864537e+06
332,NEW_YORK_CLUSTER→CHICAGO_CLUSTER,117,1.338423e+06
111,CHICAGO_CLUSTER→NEW_YORK_CLUSTER,117,1.085692e+06
571,WASHINGTON_DC_CLUSTER→NEW_YORK_CLUSTER,112,7.614511e+05


In [57]:
df = df[~df["corridor"].str.contains("OTHER_CLUSTER")]

In [58]:
# ------------------------------------------------------------
# STEP 7: FILTER CORE CORRIDORS (CLEAN + DENSE)
# ------------------------------------------------------------

MIN_FLIGHTS = 80  # tighten threshold

core_corridors = corridor_stats[
    (corridor_stats["flights"] >= MIN_FLIGHTS) &
    (~corridor_stats["corridor"].str.contains("OTHER_CLUSTER"))
]["corridor"]

df_core = df[df["corridor"].isin(core_corridors)].copy()

print("Core corridors:", len(core_corridors))
print("Rows:", len(df_core))

Core corridors: 15
Rows: 2174


In [60]:
# ------------------------------------------------------------
# STEP 8: ADD DAY OF WEEK FROM dep_datetime
# ------------------------------------------------------------

# Make sure dep_datetime is parsed correctly
df_core["dep_datetime"] = pd.to_datetime(df_core["dep_datetime"], errors="coerce")

# Build day-of-week
df_core["dow"] = df_core["dep_datetime"].dt.day_name()

# Sanity check
print("Null dep_datetime:", df_core["dep_datetime"].isna().sum())
print("Null dow:", df_core["dow"].isna().sum())

df_core[["corridor", "dep_datetime", "dow"]].head()

Null dep_datetime: 0
Null dow: 0


,corridor,dep_datetime,dow
8,NEW_YORK_CLUSTER→BOSTON_CLUSTER,2016-02-01 12:40:00,Monday
24,NEW_YORK_CLUSTER→CHICAGO_CLUSTER,2016-01-02 19:35:00,Saturday
46,MIAMI_CLUSTER→NEW_YORK_CLUSTER,2016-01-03 19:00:00,Sunday
53,MIAMI_CLUSTER→NEW_YORK_CLUSTER,2016-01-02 16:02:00,Saturday
54,MIAMI_CLUSTER→NEW_YORK_CLUSTER,2016-01-03 16:02:00,Sunday


In [61]:
# ------------------------------------------------------------
# STEP 9: BUILD CORRIDOR × DOW × REGIME TABLE
# ------------------------------------------------------------

grouped = (
    df_core.groupby(["corridor", "dow", "regime"])
           .agg(
               flights=("Trip_Legs_ID", "count"),
               median_revhr=("RevHr_clean", "median"),
               total_revenue=("Revenue_allocated", "sum")
           )
           .reset_index()
)

# Format revenue for readability
grouped["total_revenue"] = grouped["total_revenue"].round(0)

grouped.head(20)

,corridor,dow,regime,flights,median_revhr,total_revenue
0,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Friday,HIGH_VOLUME,1,3840.000000,2944.0
1,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Friday,HIGH_YIELD,2,6593.180750,6780.0
2,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Friday,NORMAL,17,5938.524590,83812.0
3,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Friday,PEAK_MIXED,9,6525.117568,49361.0
4,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Monday,HIGH_VOLUME,2,9956.737092,10353.0
5,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Monday,HIGH_YIELD,6,8377.196424,56296.0
6,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Monday,NORMAL,12,6182.772591,67000.0
7,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Monday,PEAK_MIXED,5,8681.564246,36651.0
8,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Saturday,HIGH_VOLUME,1,9844.831250,8040.0
9,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Saturday,HIGH_YIELD,6,5548.422172,33095.0


In [62]:
# ------------------------------------------------------------
# STEP 10: CHECK DENSITY OF CORRIDOR × DOW × REGIME
# ------------------------------------------------------------

print("Flights per grouped row:")
display(grouped["flights"].describe())

print("\nTop grouped rows by flights:")
display(grouped.sort_values("flights", ascending=False).head(20))

Flights per grouped row:


count    370.000000
mean       5.875676
std        5.987856
min        1.000000
25%        2.000000
50%        4.000000
75%        8.000000
max       42.000000
Name: flights, dtype: float64


Top grouped rows by flights:


,corridor,dow,regime,flights,median_revhr,total_revenue
149,MIAMI_CLUSTER→NEW_YORK_CLUSTER,Wednesday,NORMAL,42,6982.138204,662311.0
137,MIAMI_CLUSTER→NEW_YORK_CLUSTER,Sunday,NORMAL,39,6548.946903,607493.0
129,MIAMI_CLUSTER→NEW_YORK_CLUSTER,Monday,NORMAL,35,5846.907692,491577.0
248,NEW_YORK_CLUSTER→MIAMI_CLUSTER,Wednesday,NORMAL,32,5850.381266,483059.0
225,NEW_YORK_CLUSTER→MIAMI_CLUSTER,Friday,NORMAL,25,6491.167192,374643.0
240,NEW_YORK_CLUSTER→MIAMI_CLUSTER,Thursday,NORMAL,24,6593.531497,392596.0
318,SAN_FRANCISCO_CLUSTER→LOS_ANGELES_CLUSTER,Wednesday,NORMAL,24,7711.507651,182797.0
99,LOS_ANGELES_CLUSTER→SAN_FRANCISCO_CLUSTER,Wednesday,NORMAL,22,10217.914882,194207.0
228,NEW_YORK_CLUSTER→MIAMI_CLUSTER,Monday,NORMAL,22,5280.612422,303513.0
145,MIAMI_CLUSTER→NEW_YORK_CLUSTER,Tuesday,NORMAL,22,5551.403794,301360.0


In [63]:
# ------------------------------------------------------------
# STEP 11: REMOVE VERY LOW-DENSITY GROUPS
# ------------------------------------------------------------

MIN_GROUP_FLIGHTS = 8

grouped_filtered = grouped[grouped["flights"] >= MIN_GROUP_FLIGHTS].copy()

print("Rows before filter:", len(grouped))
print("Rows after filter:", len(grouped_filtered))

grouped_filtered.head(20)

Rows before filter: 370
Rows after filter: 96


,corridor,dow,regime,flights,median_revhr,total_revenue
2,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Friday,NORMAL,17,5938.524590,83812.0
3,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Friday,PEAK_MIXED,9,6525.117568,49361.0
6,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Monday,NORMAL,12,6182.772591,67000.0
10,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Saturday,NORMAL,10,6269.485588,39393.0
13,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Sunday,NORMAL,17,6127.659574,108971.0
14,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Thursday,HIGH_VOLUME,8,5860.917105,43953.0
16,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Thursday,NORMAL,12,6185.802831,161476.0
19,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Tuesday,HIGH_YIELD,14,7391.531818,106717.0
20,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Tuesday,NORMAL,12,6696.978963,75681.0
23,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Wednesday,HIGH_YIELD,8,6795.735966,41529.0


In [64]:
# ------------------------------------------------------------
# STEP 12: PIVOT FOR PRICING COMPARISON
# ------------------------------------------------------------

pivot = grouped_filtered.pivot_table(
    index=["corridor", "dow"],
    columns="regime",
    values=["flights", "median_revhr"],
    aggfunc="first"
)

# Flatten MultiIndex columns
pivot.columns = [f"{metric}_{regime}" for metric, regime in pivot.columns]
pivot = pivot.reset_index()

pivot.head()

,corridor,dow,flights_HIGH_VOLUME,flights_HIGH_YIELD,flights_NORMAL,flights_PEAK_MIXED,median_revhr_HIGH_VOLUME,median_revhr_HIGH_YIELD,median_revhr_NORMAL,median_revhr_PEAK_MIXED
0,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Friday,NaN,NaN,17.0,9.0,NaN,NaN,5938.524590,6525.117568
1,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Monday,NaN,NaN,12.0,NaN,NaN,NaN,6182.772591,NaN
2,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Saturday,NaN,NaN,10.0,NaN,NaN,NaN,6269.485588,NaN
3,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Sunday,NaN,NaN,17.0,NaN,NaN,NaN,6127.659574,NaN
4,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Thursday,8.0,NaN,12.0,NaN,5860.917105,NaN,6185.802831,NaN


In [65]:
# ------------------------------------------------------------
# STEP 13: COMPUTE LIFT + VOLUME RATIO
# ------------------------------------------------------------

# Ensure required columns exist
for col in [
    "median_revhr_NORMAL",
    "median_revhr_HIGH_YIELD",
    "flights_NORMAL",
    "flights_HIGH_YIELD"
]:
    if col not in pivot.columns:
        pivot[col] = np.nan

pivot["lift"] = pivot["median_revhr_HIGH_YIELD"] / pivot["median_revhr_NORMAL"]
pivot["volume_ratio"] = pivot["flights_HIGH_YIELD"] / pivot["flights_NORMAL"]

# Clean bad values
pivot = pivot.replace([np.inf, -np.inf], np.nan)

pivot.head()

,corridor,dow,flights_HIGH_VOLUME,flights_HIGH_YIELD,flights_NORMAL,flights_PEAK_MIXED,median_revhr_HIGH_VOLUME,median_revhr_HIGH_YIELD,median_revhr_NORMAL,median_revhr_PEAK_MIXED,lift,volume_ratio
0,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Friday,NaN,NaN,17.0,9.0,NaN,NaN,5938.524590,6525.117568,NaN,NaN
1,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Monday,NaN,NaN,12.0,NaN,NaN,NaN,6182.772591,NaN,NaN,NaN
2,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Saturday,NaN,NaN,10.0,NaN,NaN,NaN,6269.485588,NaN,NaN,NaN
3,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Sunday,NaN,NaN,17.0,NaN,NaN,NaN,6127.659574,NaN,NaN,NaN
4,BOSTON_CLUSTER→NEW_YORK_CLUSTER,Thursday,8.0,NaN,12.0,NaN,5860.917105,NaN,6185.802831,NaN,NaN,NaN
